# 01. V-JEPA 3차원 멀티블록 마스킹 기초

목표: 비디오 토큰 격자에서 여러 공간 블록을 가리고, 같은 블록을 시간축 전체에 반복하는 V-JEPA식 마스킹의 핵심을 표준 라이브러리만으로 이해합니다. 위·아래 코드 셀을 순서대로 실행하세요.

In [ ]:
from random import Random

T, H, W = 8, 14, 14  # 16프레임을 tubelet=2, patch=16으로 토큰화한 예
TOTAL = T * H * W
rng = Random(240408471)  # 재현 가능한 실습을 위한 고정 시드
print({'시간 토큰': T, '공간 격자': (H, W), '전체 토큰': TOTAL})

In [ ]:
def spatial_block(height, width, area_fraction, aspect_ratio, rng):
    # 목표 면적과 종횡비에서 블록 크기를 구한 뒤 격자 경계 안에 배치합니다.
    area = max(1, round(height * width * area_fraction))
    block_h = max(1, min(height, round((area / aspect_ratio) ** 0.5)))
    block_w = max(1, min(width, round(area / block_h)))
    top = rng.randrange(height - block_h + 1)
    left = rng.randrange(width - block_w + 1)
    return {(row, col) for row in range(top, top + block_h)
                       for col in range(left, left + block_w)}

def multi_block_mask(time, height, width, specs, rng):
    spatial = set()
    for count, fraction in specs:
        for _ in range(count):
            ratio = rng.uniform(0.75, 1.5)
            spatial |= spatial_block(height, width, fraction, ratio, rng)
    # 동일한 공간 마스크를 모든 시간 토큰에 복제해 3차원 블록을 만듭니다.
    volume = {(t, row, col) for t in range(time) for row, col in spatial}
    return spatial, volume

# 논문의 실제 샘플러를 완전히 복제하는 코드는 아니며, 두 마스크군의 아이디어를 재현합니다.
short_spatial, short_volume = multi_block_mask(T, H, W, [(8, 0.15)], rng)
long_spatial, long_volume = multi_block_mask(T, H, W, [(2, 0.70)], rng)
masked = short_volume | long_volume
print(f'가려진 토큰: {len(masked)}/{TOTAL} ({len(masked) / TOTAL:.1%})')

In [ ]:
def draw_slice(mask, time_index=0):
    return '\n'.join(
        ''.join('##' if (time_index, row, col) in mask else '..' for col in range(W))
        for row in range(H)
    )

print(draw_slice(masked))
# 시간축 전체에 같은 공간 블록이 반복되는지 확인합니다.
slice_zero = {(row, col) for t, row, col in masked if t == 0}
for t in range(1, T):
    assert slice_zero == {(row, col) for tt, row, col in masked if tt == t}
assert 0 < len(masked) < TOTAL
print('검증 완료: 모든 시간 슬라이스의 공간 마스크가 같습니다.')

## 해석

블록이 겹치므로 단순히 `개수 × 비율`을 더한 값과 실제 마스킹 비율은 다릅니다. 논문은 짧은 범위와 긴 범위의 블록을 섞어 평균 약 90%를 가리며, 인코더가 보지 못한 시공간 영역의 표현을 문맥에서 예측하도록 만듭니다.